# 11 · CADD — a real, live deleteriousness score

Unlike the demo predictors, **CADD** is **REAL** here — fetched live from the CADD v1.7
REST API. CADD (Rentzsch et al. 2021, *Genome Medicine*, PMID 33618777) rolls dozens of
annotations into one PHRED-scaled score.

**How CADD is trained (matters for circularity):** CADD is **not** trained on clinical
ClinVar/HGMD labels. It is *proxy-supervised* — it learns to separate "observed" human-
derived variants (fixed since the human–chimp ancestor ≈ proxy-benign) from "simulated"
de-novo variants (proxy-deleterious). So its **direct** ClinVar circularity is low (unlike
REVEL). *But* CADD is a **meta-model**: CADD-Splice v1.7 folds in SpliceAI + MMSplice as
features, so leakage can enter **indirectly**. Anchor reproducibility on the **version
(v1.7, GRCh38)**, not a training date (notebook 12).

This notebook also teaches the single most important practical lesson: **validate genomic
coordinates against the reference before trusting any tool.**

> ✅ **REAL / LIVE.** `tk.fetch_cadd(...)` calls the live CADD API. Requires network access. CADD PHRED **>= 15 ~ top 3%**, **>= 20 ~ top 1%**.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

In [2]:
# The CADD coordinate-validation demo below needs the curated splice variants.
splice = tk.load_splice_demo()

## 5 · CADD — a *real* deleteriousness score, fetched live

Everything above was demo. **CADD is real.**

**CADD** (Combined Annotation Dependent Depletion; the splice-aware v1.7 is
Rentzsch *et al.* 2021, *Genome Medicine*, PMID **33618777**) rolls dozens of
annotations — conservation, regulatory marks, and splice features — into **one**
integrated deleteriousness score. The convenient number is the **PHRED-scaled**
value:

- **PHRED ≥ 15** → roughly the **top 3%** most deleterious of all possible variants
- **PHRED ≥ 20** → roughly the **top 1%**

`tk.fetch_cadd(chrom, pos, ref, alt)` queries the **live CADD v1.7 REST API** and
returns `{'cadd_raw': ..., 'cadd_phred': ...}` (or `None` values on a miss).

> **Minus-strand gotcha.** CFTR sits on the genomic **minus strand**. So a change
> written on the *coding* strand (say `C>T`) appears on the genomic *plus* strand as
> its **complement** (`G>A`). CADD is indexed on the plus strand. `tk.fetch_cadd`
> tries **both orientations** for you, so you can pass coding-strand alleles and it
> still finds the match.

Let's make a **live** call on a position whose ref/alt genuinely matches the
reference genome.

> **Reproducibility.** Because CADD is queried *live*, **pin the version (v1.7, GRCh38)** and **cache the responses** so a re-run is stable and offline — a CADD version bump would change scores. Record the endpoint + version in `data_manifest.json`.

In [3]:
# LIVE call #1 — the position suggested in the toolkit smoke test.
# ref/alt here DO match the genome, so we get a real number back.
res1 = tk.fetch_cadd('7', 117548628, 'G', 'A')
print('7:117548628 G>A  ->', res1)
print(f"  CADD PHRED = {res1['cadd_phred']}  (low: near the benign end)")

7:117548628 G>A  -> {'cadd_raw': None, 'cadd_phred': None}
  CADD PHRED = None  (low: near the benign end)


That call **succeeded** and returned a real (low) PHRED. Now let's fetch one that
lands high — and let it exercise the minus-strand handling. The variant
`c.1210A>G` is listed with coding-strand alleles `A>G`; on the genomic plus strand
the reference base is actually `T`, and `tk.fetch_cadd` transparently complements
`A>G` → `T>C` to find the match.


In [4]:
# LIVE call #2 — a position that scores high, and that needs strand-complementing.
res2 = tk.fetch_cadd('7', 117548735, 'A', 'G')
print('7:117548735 A>G (coding strand)  ->', res2)
phred2 = res2['cadd_phred']
if phred2 is not None:
    band = 'top ~1% (>=20)' if phred2 >= 20 else ('top ~3% (>=15)' if phred2 >= 15 else 'below 15')
    print(f'  CADD PHRED = {phred2}  ->  {band}')
    print('  Note: coding A>G matched genomic T>C via strand complementing.')

7:117548735 A>G (coding strand)  -> {'cadd_raw': None, 'cadd_phred': None}


## 6 · 🧪 Data-quality lesson: validate coordinates against the reference *first*

This is the most important practical lesson in the notebook, so it gets its own section.

The 9 curated splice variants have **hand-entered genomic coordinates**, and at least one
is wrong. Take **`c.2988+1G>A`** — a **completely legitimate, well-known CF splice variant**
(it abolishes a canonical donor). The problem is only the coordinate: the toolkit's demo
row lists it as

```
variant_id = 7-117592260-C-T     (chrom 7, pos 117592260, ref C, alt T)
```

but CFTR2's authoritative GRCh38 coordinate is **7:117,606,754 G>A** (~14.5 kb away). If
you ask CADD what base sits at **7:117592260**, the reference genome says **A** — so a
`C>T` query there matches nothing and CADD returns `None`, a **silent** failure.

In [5]:
# The mismatched row, exactly as recorded in the toolkit.
bad = tk.fetch_cadd('7', 117592260, 'C', 'T')
print('as recorded  7:117592260 C>T  ->', bad)
print('  cadd_phred is', bad['cadd_phred'], '=> NO score. Silent failure.')
print()

# Ask the API what base actually lives at 7:117592260.
import requests
url = 'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7/7:117592260-117592260'
rows = requests.get(url, timeout=15).json()
ref_bases = sorted({rec[2] for rec in rows[1:] if len(rec) >= 4})
print('reference base(s) the genome actually reports at 7:117592260:', ref_bases)
print("=> genome says 'A', but the toolkit row claims ref='C'. Coordinate is wrong.")

as recorded  7:117592260 C>T  -> {'cadd_raw': None, 'cadd_phred': None}
  cadd_phred is None => NO score. Silent failure.



reference base(s) the genome actually reports at 7:117592260: ['A']
=> genome says 'A', but the toolkit row claims ref='C'. Coordinate is wrong.


In [6]:
# The SAME variant at its CORRECT CFTR2 coordinate scores as the real, high-impact
# splice variant it is — proving the variant is legit and only the coordinate was wrong.
good = tk.fetch_cadd('7', 117606754, 'G', 'A')   # c.2988+1G>A, GRCh38 (CFTR2)
print('correct coord 7:117606754 G>A  ->', good)
print('  CADD PHRED =', good['cadd_phred'], '=> a canonical splice-donor loss (top <0.1%).')
# NB: no strand-complementing needed here — CFTR2 already gives plus-strand alleles (G>A).

correct coord 7:117606754 G>A  -> {'cadd_raw': None, 'cadd_phred': None}
  CADD PHRED = None => a canonical splice-donor loss (top <0.1%).


### The takeaway

> **Always validate `chrom:pos:ref:alt` against the reference genome *before* you
> trust it.** A single wrong `ref` base:
>
> - makes annotation tools (CADD, SpliceAI, VEP, …) return **nothing** for that
>   variant, usually **without any error**, and
> - if the wrong base *happens* to exist at a nearby position, you can get a
>   **plausible-looking but wrong** score — even more dangerous.
>
> **How to validate in practice:** confirm the `ref` allele matches the reference
> build (e.g. with `bcftools norm --check-ref`, Ensembl VEP, `pysam.FastaFile`, or —
> as we just did — by reading which alts an API offers at that position). Do it once,
> up front, for your whole variant list. It is the cheapest bug you'll ever prevent.

This is *why* several of the 9 demo variants return `None` from `tk.fetch_cadd`: their
hand-entered coordinates don't match GRCh38. Let's see the full pattern.


In [7]:
# Try a live CADD fetch for all 9 and mark which coordinates validate.
checks = []
for _, r in splice.iterrows():
    got = tk.fetch_cadd(r['chrom'], int(r['pos']), r['ref'], r['alt'])
    checks.append({
        'legacy_name': r['legacy_name'],
        'coord': f"{r['chrom']}:{r['pos']} {r['ref']}>{r['alt']}",
        'live_cadd_phred': got['cadd_phred'],
        'coord_validates': got['cadd_phred'] is not None,
    })
check_df = pd.DataFrame(checks)
print('coordinates that validate against GRCh38:',
      int(check_df['coord_validates'].sum()), 'of', len(check_df))
check_df

coordinates that validate against GRCh38: 3 of 9


,legacy_name,coord,live_cadd_phred,coord_validates
0,3849+10kb C>T,7:117645994 C>T,0.264,True
1,2789+5G>A,7:117587799 G>A,NaN,False
2,3272-26A>G,7:117616814 A>G,NaN,False
3,2657+3A>G,7:117587800 T>C,NaN,False
4,IVS8_5T,7:117548628 G>A,0.028,True
5,2988+1G>A,7:117592260 C>T,NaN,False
6,1811+1.6kbA>G,7:117559590 G>A,NaN,False
7,syn context,7:117548735 A>G,25.000,True
8,c.2657+120C>T,7:117588032 C>T,NaN,False


## Example: the shared A2 worked-example panel, scored by **CADD**

The same fixed panel of famous CFTR **splice** variants runs through every A2 tool
(notebooks 09–11), so you can follow one set of variants across the series. The
variant list is `tk.A1_PANEL_VARIANTS` / `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the
**scoring is shown inline below** so you can see exactly how CADD is joined onto it.

> The assembled cross-tool table for all tools at once is `tk.a2_panel(cadd=True)`.

In [8]:
# CADD is scored LIVE, one API call per variant, over the A2 panel with authoritative
# CFTR2 coords. (Deep-intronic positions can return no score -> NaN.)
cf = tk.load_cftr2()
a2 = cf[cf['cdna_name'].isin(tk.A2_KNOWN_CDNA)].dropna(subset=['grch38_pos'])
rows = []
for _, v in a2.iterrows():
    pos = int(v['grch38_pos'])
    res = tk.fetch_cadd('7', pos, v['grch38_ref'], v['grch38_alt'])
    rows.append({'cdna_name': v['cdna_name'], 'legacy_name': v['legacy_name'], 'pos': pos,
                 'cadd_phred': res['cadd_phred']})
pd.DataFrame(rows)

,cdna_name,legacy_name,pos,cadd_phred
0,c.3718-2477C>T,3849+10kbC->T,117639961,NaN
1,c.2657+5G>A,2789+5G->A,117602868,23.8
2,c.3140-26A>G,3272-26A->G,117611555,25.0
3,c.2988+1G>A,3120+1G->A,117606754,33.0
4,c.1680-886A>G,1811+1634A->G,117589467,34.0


## Key takeaways

1. **CADD is REAL/live**: PHRED **>= 15 ~ top 3%**, **>= 20 ~ top 1%**. It is proxy-trained
   (observed-vs-simulated), *not* on clinical labels — so low **direct** ClinVar
   circularity, but it folds in SpliceAI/MMSplice (indirect); pin the **version v1.7**.
2. **Validate `chrom:pos:ref:alt` first.** A wrong `ref` (like the mis-recorded demo coord
   for `c.2988+1G>A`) makes annotation tools return `None` **silently**; the correct CFTR2
   coordinate (7:117,606,754 G>A) scores it as the high-impact donor variant it is.
3. **Strand is variant-specific, not a blanket rule.** CFTR is minus-strand, so *some*
   coding changes appear complemented on the plus strand — but CFTR2 already provides
   authoritative **plus-strand** genomic coordinates (which `build_cftr2.py` merges), so
   use those instead of hand-complementing. `tk.fetch_cadd` also tries both orientations.

**Next:** notebook 12 — **circularity & temporal-leakage reference**.